# RDDT V4 confirmed amyloidosis extraction

This is the urgent confirmed-only Snowflake Workspace runner. It reports the enabled CLAIMS source count, then extracts confirmed amyloidosis patients from CLAIMS ICD-10 codes only. Detection does not use non-CLAIMS tables, dates, ICD-9, NLP, spaCy, or medSpaCy. After the patient IDs are confirmed, a separate read-only hydration phase retrieves their complete EHR from every table marked `profile_enabled`. The notebook creates no Snowflake table, view, or stage.

In [ ]:
from pathlib import Path
import json
import sys

package_candidates = [Path.cwd() / 'v4', Path.cwd(), Path.cwd().parent / 'v4']
package_dir = next((p.resolve() for p in package_candidates if (p / '__init__.py').exists() and (p / 'config').is_dir()), None)
if package_dir is None:
    raise RuntimeError('Run this notebook from the v4 folder or its parent directory.')
project_root = package_dir.parent
sys.path.insert(0, str(project_root))

from snowflake.snowpark.context import get_active_session
from v4.output.patient_profile import build_patient_profile, export_confirmed_icd10_profiles
from v4.warehouse.source_schema import default_source_config, is_table_profile_enabled, is_table_profile_required, qualified_table_name, quote_identifier

config_dir = package_dir / 'config'
export_dir = Path.cwd() / 'rddt_v4_exports'
session = get_active_session()
print({'execution': 'CONFIRMED_ONLY', 'source_mode': 'CLAIMS_ONLY', 'terminology_mode': 'ICD10_ONLY', 'dates_required': False, 'suspicion_pipeline': 'NOT_RUN', 'nlp': 'DISABLED', 'warehouse_runtime': 'REQUIRED'})

## Physical source location

The active namespace comes from `config/source_schema.json`. Leave `SOURCE_NAMESPACE_OVERRIDE` as `None` to use that profile value. Set the override only for a deliberate deployment-specific namespace. The first run requires CLAIMS as the sole algorithm-enabled table; the independent `profile_enabled` flags identify which available EHR tables are read after confirmation.

In [ ]:
source_config = default_source_config()
SOURCE_NAMESPACE_OVERRIDE = None  # Optional: 'DATABASE.SCHEMA'; None preserves the profile namespace
if SOURCE_NAMESPACE_OVERRIDE is not None:
    source_config['namespace'] = SOURCE_NAMESPACE_OVERRIDE
enabled_tables = [key for key, table in source_config.get('tables', {}).items() if table.get('enabled')]
if enabled_tables != ['claim']:
    raise ValueError(f'First run must enable CLAIMS only; enabled tables are {enabled_tables!r}')
print({'source_namespace': source_config.get('namespace'), 'enabled_tables': enabled_tables})

# Gate 1: summarize only tables explicitly enabled by source_schema.json.
# Disabled mappings remain preserved in config but are never queried here.
enabled_source_tables = {key: table for key, table in source_config.get('tables', {}).items() if table.get('enabled')}
analysis_summary = []
for table_key, table_cfg in enabled_source_tables.items():
    table_name = str(table_cfg['name'])
    patient_column = table_cfg.get('columns', {}).get('patient_id')
    if not patient_column:
        raise RuntimeError(f'Enabled source {table_key!r} has no patient_id mapping.')
    table_sql = qualified_table_name(table_cfg, str(source_config.get('namespace')) if source_config.get('namespace') else None)
    row = session.sql(f"SELECT COUNT(*) AS TOTAL_ROWS, COUNT(DISTINCT {quote_identifier(patient_column)}) AS DISTINCT_PATIENTS FROM {table_sql}").collect()[0].as_dict()
    analysis_summary.append({'table_key': table_key, 'table_name': table_name, 'status': 'OK', 'total_rows': row.get('TOTAL_ROWS'), 'distinct_patients': row.get('DISTINCT_PATIENTS')})
import pandas as pd
from IPython.display import display
analysis_summary_df = pd.DataFrame(analysis_summary)
display(analysis_summary_df)

# Gate 2: confirmed-only, ICD-10-only extraction. This intentionally does
# not use event dates, screening cutoffs, NLP, or the suspicion pipeline.
confirmed_config = json.loads((config_dir / 'shared' / 'confirmed_patients.json').read_text(encoding='utf-8'))
route = confirmed_config.get('routes', {}).get('ALL_AMYLOIDOSIS', {})
icd10_terms = [row for row in route.get('terminology', []) if str(row.get('terminology_system', '')).upper().replace('-', '').replace('_', '') in {'ICD10', 'ICD10CM'}]
if not icd10_terms:
    raise RuntimeError('ALL_AMYLOIDOSIS has no configured ICD-10 terminology; stop.')
claim_cfg = source_config['tables']['claim']
claim_table_sql = qualified_table_name(claim_cfg, str(source_config.get('namespace')) if source_config.get('namespace') else None)
claim_cols = claim_cfg.get('columns', {})
patient_col = claim_cols.get('patient_id')
type_col = claim_cols.get('diagnosis_type')
code_col = claim_cols.get('diagnosis_code')
if not patient_col or not type_col or not code_col:
    raise RuntimeError('CLAIMS ICD-10 mapping must include patient_id, diagnosis_type, and diagnosis_code.')
run_id = __import__('uuid').uuid4().hex
config_hash = __import__('hashlib').sha256(json.dumps(confirmed_config, sort_keys=True, separators=(',', ':')).encode('utf-8')).hexdigest()
def _sql_literal(value):
    return "'" + str(value).replace("'", "''") + "'"
type_predicate = f"REGEXP_REPLACE(UPPER(TRIM(CAST({quote_identifier(type_col)} AS VARCHAR))), '[^A-Z0-9]', '') IN ('ICD10', 'ICD10CM')"
config_rows = []
for term in icd10_terms:
    dotted = str(term.get('value', '')).strip().upper()
    normalized = ''.join(character for character in dotted if character.isalnum())
    if not normalized:
        continue
    config_rows.append('(' + ', '.join(_sql_literal(value) for value in (normalized, dotted, term.get('amyloidosis_type', 'AMYLOIDOSIS'), term.get('subtype_label', dotted), term.get('risk_label', 'CONFIRMED'), term.get('priority_label', 'CONFIRMED'), term.get('confirmation_scope', 'KNOWN_AMYLOIDOSIS'))) + ')')
if not config_rows:
    raise RuntimeError('ALL_AMYLOIDOSIS contains no usable ICD-10 codes; stop.')
config_values_sql = ', '.join(config_rows)
confirmed_profile_sql = f"""WITH CONFIG_CODES AS (
    SELECT COLUMN1 AS NORMALIZED_CODE, COLUMN2 AS MATCHED_CODE, COLUMN3 AS AMYLOIDOSIS_TYPE, COLUMN4 AS AMYLOIDOSIS_SUBTYPE, COLUMN5 AS RISK_LABEL, COLUMN6 AS PRIORITY_LABEL, COLUMN7 AS CONFIRMATION_SCOPE
    FROM VALUES {config_values_sql}
), SOURCE_CODES AS (
    SELECT CAST({quote_identifier(patient_col)} AS VARCHAR) AS PATIENT_ID, REGEXP_REPLACE(UPPER(TRIM(CAST({quote_identifier(code_col)} AS VARCHAR))), '[^A-Z0-9]', '') AS NORMALIZED_CODE
    FROM {claim_table_sql}
    WHERE {quote_identifier(patient_col)} IS NOT NULL AND {quote_identifier(code_col)} IS NOT NULL AND {type_predicate}
), CONFIRMED_MATCHES AS (
    SELECT DISTINCT '{run_id}' AS RUN_ID, S.PATIENT_ID, C.MATCHED_CODE, C.AMYLOIDOSIS_TYPE, C.AMYLOIDOSIS_SUBTYPE, C.RISK_LABEL, C.PRIORITY_LABEL, C.CONFIRMATION_SCOPE, '{config_hash}' AS CONFIG_HASH
    FROM SOURCE_CODES S JOIN CONFIG_CODES C ON S.NORMALIZED_CODE = C.NORMALIZED_CODE
)
SELECT RUN_ID, PATIENT_ID, 'CONFIRMED' AS STATUS,
       LISTAGG(DISTINCT MATCHED_CODE, ',') WITHIN GROUP (ORDER BY MATCHED_CODE) AS MATCHED_ICD10_CODES,
       LISTAGG(DISTINCT AMYLOIDOSIS_TYPE, ',') WITHIN GROUP (ORDER BY AMYLOIDOSIS_TYPE) AS AMYLOIDOSIS_TYPES,
       LISTAGG(DISTINCT AMYLOIDOSIS_SUBTYPE, ' | ') WITHIN GROUP (ORDER BY AMYLOIDOSIS_SUBTYPE) AS AMYLOIDOSIS_SUBTYPES,
       LISTAGG(DISTINCT CONFIRMATION_SCOPE, ',') WITHIN GROUP (ORDER BY CONFIRMATION_SCOPE) AS CONFIRMATION_SCOPES,
       MAX(RISK_LABEL) AS RISK_LABEL, MAX(PRIORITY_LABEL) AS PRIORITY_LABEL,
       TRUE AS EXCLUDED_FROM_SUSPICION, FALSE AS DATE_REQUIRED, MAX(CONFIG_HASH) AS CONFIG_HASH
FROM CONFIRMED_MATCHES
GROUP BY RUN_ID, PATIENT_ID
"""
confirmed_profiles_df = session.sql(confirmed_profile_sql).to_pandas()
display(confirmed_profiles_df)

# Confirmation is decided from CLAIMS only. Profile hydration is a separate
# read-only phase that retrieves every native row from every available EHR
# table marked profile_enabled in source_schema.json.
confirmed_patient_ids_sql = f"""WITH CONFIG_CODES AS (
    SELECT COLUMN1 AS NORMALIZED_CODE FROM VALUES {config_values_sql}
), SOURCE_CODES AS (
    SELECT CAST({quote_identifier(patient_col)} AS VARCHAR) AS PATIENT_ID,
           REGEXP_REPLACE(UPPER(TRIM(CAST({quote_identifier(code_col)} AS VARCHAR))), '[^A-Z0-9]', '') AS NORMALIZED_CODE
    FROM {claim_table_sql}
    WHERE {quote_identifier(patient_col)} IS NOT NULL AND {quote_identifier(code_col)} IS NOT NULL AND {type_predicate}
)
SELECT DISTINCT S.PATIENT_ID FROM SOURCE_CODES S JOIN CONFIG_CODES C ON S.NORMALIZED_CODE = C.NORMALIZED_CODE
"""

source_object_items = []
for logical_name, physical_name in claim_cols.items():
    source_object_items.extend((_sql_literal(logical_name), f'SRC.{quote_identifier(physical_name)}'))
source_claim_object_sql = 'OBJECT_CONSTRUCT_KEEP_NULL(' + ', '.join(source_object_items) + ')'
confirmation_evidence_sql = f"""WITH CONFIG_CODES AS (
    SELECT COLUMN1 AS NORMALIZED_CODE, COLUMN2 AS MATCHED_CODE, COLUMN3 AS AMYLOIDOSIS_TYPE, COLUMN4 AS AMYLOIDOSIS_SUBTYPE, COLUMN5 AS RISK_LABEL, COLUMN6 AS PRIORITY_LABEL, COLUMN7 AS CONFIRMATION_SCOPE
    FROM VALUES {config_values_sql}
)
SELECT CAST(SRC.{quote_identifier(patient_col)} AS VARCHAR) AS PATIENT_ID,
       REGEXP_REPLACE(UPPER(TRIM(CAST(SRC.{quote_identifier(type_col)} AS VARCHAR))), '[^A-Z0-9]', '') AS DIAGNOSIS_SYSTEM,
       CC.MATCHED_CODE, CC.AMYLOIDOSIS_TYPE, CC.AMYLOIDOSIS_SUBTYPE,
       CC.RISK_LABEL, CC.PRIORITY_LABEL, CC.CONFIRMATION_SCOPE,
       {source_claim_object_sql} AS SOURCE_CLAIM
FROM {claim_table_sql} SRC
JOIN CONFIG_CODES CC
  ON REGEXP_REPLACE(UPPER(TRIM(CAST(SRC.{quote_identifier(code_col)} AS VARCHAR))), '[^A-Z0-9]', '') = CC.NORMALIZED_CODE
WHERE {type_predicate}
ORDER BY PATIENT_ID
"""

summary_by_patient = {str(row['PATIENT_ID']): row for row in confirmed_profiles_df.to_dict(orient='records')}
confirmed_patient_ids = list(summary_by_patient)
confirmation_evidence_by_patient = {patient_id: [] for patient_id in confirmed_patient_ids}
for row in session.sql(confirmation_evidence_sql).to_local_iterator():
    payload = row.as_dict() if hasattr(row, 'as_dict') else dict(row)
    patient_id = str(payload.get('PATIENT_ID'))
    if patient_id in confirmation_evidence_by_patient:
        confirmation_evidence_by_patient[patient_id].append(payload)

ehr_records_by_patient = {patient_id: {} for patient_id in confirmed_patient_ids}
source_coverage = {}
profile_tables_queried = []
for table_key, table_cfg in source_config.get('tables', {}).items():
    if not is_table_profile_enabled(table_cfg):
        source_coverage[table_key] = {'status': 'NOT_AVAILABLE_IN_CURRENT_WAREHOUSE' if not table_cfg.get('columns') else 'PROFILE_DISABLED', 'physical_table': table_cfg.get('name')}
        continue
    patient_column = table_cfg.get('columns', {}).get('patient_id')
    if not patient_column:
        raise RuntimeError(f'Profile-enabled source {table_key!r} has no patient_id mapping.')
    physical_table = qualified_table_name(table_cfg, str(source_config.get('namespace')) if source_config.get('namespace') else None)
    hydrate_sql = f"""WITH CONFIRMED_PATIENTS AS ({confirmed_patient_ids_sql})
SELECT CAST(SRC.{quote_identifier(patient_column)} AS VARCHAR) AS PROFILE_PATIENT_ID, SRC.*
FROM {physical_table} SRC
JOIN CONFIRMED_PATIENTS CP ON CAST(SRC.{quote_identifier(patient_column)} AS VARCHAR) = CP.PATIENT_ID
ORDER BY PROFILE_PATIENT_ID
"""
    try:
        for row in session.sql(hydrate_sql).to_local_iterator():
            payload = row.as_dict() if hasattr(row, 'as_dict') else dict(row)
            patient_id = str(payload.pop('PROFILE_PATIENT_ID'))
            if patient_id in ehr_records_by_patient:
                ehr_records_by_patient[patient_id].setdefault(table_key, []).append(payload)
        for patient_id in confirmed_patient_ids:
            ehr_records_by_patient[patient_id].setdefault(table_key, [])
        source_coverage[table_key] = {'status': 'QUERIED', 'physical_table': table_cfg.get('name')}
        profile_tables_queried.append(table_key)
    except Exception as exc:
        source_coverage[table_key] = {'status': 'UNAVAILABLE', 'physical_table': table_cfg.get('name'), 'error': str(exc)}
        if is_table_profile_required(table_cfg):
            raise RuntimeError(f'Required EHR profile source {table_key!r} could not be hydrated: {exc}') from exc

def _mapped_value(row, physical_name):
    return next((value for key, value in row.items() if str(key).strip('\"').upper() == str(physical_name).strip('\"').upper()), None)

confirmed_patient_profiles = []
census_cfg = source_config.get('tables', {}).get('census', {})
for patient_id, summary in summary_by_patient.items():
    census_rows = ehr_records_by_patient[patient_id].get('census', [])
    demographics = ({logical: _mapped_value(census_rows[0], physical) for logical, physical in census_cfg.get('columns', {}).items() if logical != 'patient_id'} if census_rows else {})
    confirmed_patient_profiles.append(build_patient_profile(
        patient_id, confirmation_summary=summary, confirmation_evidence=confirmation_evidence_by_patient[patient_id],
        ehr_records_by_table=ehr_records_by_patient[patient_id], source_config=source_config, source_coverage=source_coverage,
        demographics=demographics, run_id=run_id, config_hash=config_hash, confirmation_source_table=str(claim_cfg['name'])
    ))
if len(confirmed_patient_profiles) != len(confirmed_profiles_df):
    raise RuntimeError(f'Profile count {len(confirmed_patient_profiles)} does not match confirmed summary count {len(confirmed_profiles_df)}.')
print({'confirmed_profiles': len(confirmed_profiles_df), 'complete_json_profiles': len(confirmed_patient_profiles), 'detection_tables': enabled_tables, 'ehr_profile_tables': profile_tables_queried, 'snowflake_objects_created': [], 'suspicion_pipeline': 'NOT_RUN', 'date_required': False})

## Review and download

This first-run notebook stops after confirmed-only detection. `confirmed_profiles_df` is the compact one-row-per-patient index. Detection uses CLAIMS ICD-10 only. After detection, the profile-hydration phase reads every native row for each confirmed patient from all `profile_enabled` EHR tables: CENSUS, CLAIMS, ENCOUNTERS, LABS, MEDICATIONS, and SURGICAL_HISTORY in the current schema. The four tables absent from this warehouse are recorded as unavailable and are not queried. `confirmed_patient_profiles` is produced by the same canonical builder used by the suspicion pipeline and contains the full EHR, exact confirmation evidence, phenotype-specific risk, and algorithm path. All queries are read-only and create no Snowflake object. Dates are retained when present but are not required for confirmation.

In [ ]:
from IPython.display import FileLink, display
export_dir.mkdir(parents=True, exist_ok=True)
summary_path = export_dir / 'confirmed_amyloidosis_profiles.csv'
confirmed_profiles_df.to_csv(summary_path, index=False)
profile_exports = export_confirmed_icd10_profiles(confirmed_patient_profiles, export_dir)
display(FileLink(summary_path))
display(FileLink(profile_exports['jsonl']))
display(FileLink(profile_exports['csv']))
print({'summary_csv': str(summary_path), 'complete_profile_jsonl': profile_exports['jsonl'], 'complete_profile_csv_index': profile_exports['csv'], 'snowflake_objects_created': [], 'workspace_files_only': True})

## Audit-only outputs

The confirmed-only first run creates no Snowflake result objects. Review `confirmed_patient_profiles` in the session or download `confirmed/confirmed_amyloidosis_patient_profiles.jsonl`. Each JSONL line is one complete patient profile containing the patient's full available EHR grouped by table, source coverage/completeness, exact confirmation evidence, phenotype-specific risk, and the confirmed pre-screen algorithm path. Download the workspace-local files before ending the session.

In [ ]:
print({'analysis_summary_rows': len(analysis_summary_df), 'confirmed_summary_rows': len(confirmed_profiles_df), 'complete_json_profiles': len(confirmed_patient_profiles), 'complete_profile_jsonl': profile_exports['jsonl'], 'suspicion_pipeline': 'NOT_RUN', 'date_required_for_confirmed_run': False})